# Model Evaluation and Comparison Notebook

**Objective:** This notebook is used to evaluate the performance of fine-tuned models (SFT, TVAFT, ReFT) and compare them with the base model (PhoGPT-4B-Chat) on a consistent validation dataset.

**Procedure:**
1. **Install & Load Libraries:** Install the libraries needed for evaluation.

2. **Load Data:** Load the `hungnm/vietnamese-medical-qa` dataset and split it into a fixed validation set (using `seed=42`).

3. **Define Utility Functions:** Includes functions to load the model, generate answers, and calculate metrics.

4. **Run Evaluation:** Run each model in turn on the validation set to generate answers and calculate metrics.

5. **Display & Save Results:** Present the results in a comparison table and save them as a CSV file.

In [ ]:
# Cell 1: Install the necessary libraries
!pip install -q "transformers==4.41.2" "peft==0.10.0" "accelerate==0.29.3" "datasets==2.19.0" \
    "bitsandbytes>=0.43.2" "trl==0.8.6" "evaluate" "rouge_score" "bert_score" \
    "sacrebleu" "pyvi" pandas pyyaml

In [ ]:
# Cell 2: Import library and add path to src
import os
import sys
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
from datasets import load_dataset, DatasetDict
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Add the root directory to the path so that it can be imported from src
sys.path.append('..')
from src.utils.metrics import compute_all_metrics

## 1. Configuration and Data Loading

In [ ]:
# Cell 3: Load configuration and define constants
with open('../src/configs/base_config.yaml', 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

BASE_MODEL_NAME = config['model']['base_model_name']
SFT_ADAPTER_PATH = "../src/models/phogpt-4b-medical-chatbot-sft"
TVAFT_ADAPTER_PATH = "../src/models/phogpt-4b-medical-chatbot-tvaft"
REFT_ADAPTER_PATH = "../src/models/phogpt-4b-medical-chatbot-reft"

N_SAMPLES_TO_EVALUATE = 933
SEED = config['data']['seed']
TEST_SIZE = config['data']['test_size']
DATASET_NAME = config['data']['dataset_name']

In [ ]:
# Cell 4: Load and prepare consistent validation dataset
print(f"Loading dataset '{DATASET_NAME}'...")
full_dataset = load_dataset(DATASET_NAME, split="train")
train_test_split = full_dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
validation_dataset = train_test_split['test']

print(f"Use {len(validation_dataset)} samples in the validation set.")
print(f"Will run evaluation on the first {N_SAMPLES_TO_EVALUATE} samples.")

## 2. Utility Functions

In [ ]:
# Cell 5: Function to load model and create pipeline
def load_model_and_pipeline(base_model_name, adapter_path=None, quant_config=None):
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.float16

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

    print(f"Loading base model: {base_model_name}")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if adapter_path and os.path.isdir(adapter_path):
        print(f"Currently applying LoRA adapter from: {adapter_path}")
        model = PeftModel.from_pretrained(base_model, adapter_path)
    else:
        if adapter_path:
            print(f"Warning: Adapter directory at '{adapter_path}' not found. Using base model.")
        model = base_model

    model.eval()

    qa_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    return qa_pipeline

In [ ]:
# Cell 6: The function that generates the answer
def generate_predictions(qa_pipeline, dataset, n_samples):
    predictions, references = [], []
    prompt_template = config['data']['prompt_template']
    
    for i in tqdm(range(n_samples), desc="Đang sinh câu trả lời"):
        sample = dataset[i]
        question = sample['question']
        reference_answer = sample['answer']
        input_text = prompt_template.format(instruction=question)
        
        full_output = qa_pipeline(input_text)[0]["generated_text"]
        
        try:
            model_answer = full_output.split("### Trả lời:")[1].strip()
        except IndexError:
            model_answer = ""
            
        predictions.append(model_answer)
        references.append(reference_answer)
        
    return predictions, references

## 3. Download Models and Run Evaluation

In [ ]:
# Cell 7: Define the models to be evaluated
models_to_evaluate = {
    "Base Model (PhoGPT-4B)": {"adapter_path": None},
    "SFT Fine-tuned": {"adapter_path": SFT_ADAPTER_PATH},
    "ReFT (DPO) Fine-tuned": {"adapter_path": REFT_ADAPTER_PATH},
    "TVAFT Fine-tuned": {"adapter_path": TVAFT_ADAPTER_PATH},
}

In [ ]:
# Cell 8: Main loop to run evaluation
all_results = {}
all_predictions = {}

for model_name, paths in models_to_evaluate.items():
    print(f"\n{'='*20} MODEL IN REVIEW: {model_name.upper()} {'='*20}")
    
    pipe = load_model_and_pipeline(BASE_MODEL_NAME, adapter_path=paths['adapter_path'])
    
    predictions, references = generate_predictions(pipe, validation_dataset, n_samples=N_SAMPLES_TO_EVALUATE)
    all_predictions[model_name] = predictions
    
    metrics = compute_all_metrics(predictions, references)
    all_results[model_name] = metrics
    
    del pipe
    torch.cuda.empty_cache()

## 4. Display and Save Results

In [ ]:
# Cell 9: Process and display the results table
comparison_df = pd.DataFrame(all_results).reset_index()
comparison_df.rename(columns={'index': 'Metric'}, inplace=True)

def highlight_best(df):
    df_highlighted = df.copy()
    for i, row in df.iterrows():
        metric_name = row['Metric']
        numeric_values = pd.to_numeric(row[1:], errors='coerce')
        if numeric_values.notna().any():
            max_val = numeric_values.max()
            for col in df.columns[1:]:
                val = pd.to_numeric(df_highlighted.loc[i, col], errors='coerce')
                if val == max_val:
                    df_highlighted.loc[i, col] = f"**{val:.2f}**"
                else:
                    df_highlighted.loc[i, col] = f"{val:.2f}"
    return df_highlighted

formatted_df = highlight_best(comparison_df)

print("\n" + "="*30 + " QUANTITATIVE RESULTS COMPARISON TABLE " + "="*30)
display(formatted_df)
print("="*95)

# Lưu kết quả
os.makedirs("../reports", exist_ok=True)
output_csv_path = "../reports/quantitative_comparison_results.csv"
comparison_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
print(f"\nSaved detailed results table to: {output_csv_path}")

## 5. Qualitative Analysis

Check out some specific examples to see the difference in the quality of answers.

In [ ]:
# Cell 10: Shows some examples for comparison
qualitative_df = pd.DataFrame({
    'Question': [sample['question'] for sample in validation_dataset.select(range(5))],
    'Reference Answer': [sample['answer'] for sample in validation_dataset.select(range(5))]
})

for model_name in models_to_evaluate.keys():
    qualitative_df[f'{model_name} Prediction'] = all_predictions[model_name][:5]

print("--- QUALITATIVE COMPARISON OF THE FIRST 5 SAMPLES ---")
for i in range(5):
    print(f"\n{'='*20} EXAMPLE {i+1} {'='*20}")
    print(f"❓ QUESTION: {qualitative_df.loc[i, 'Question']}")
    print("-"*50)
    print(f"✅ REFERENCE ANSWER:\n{qualitative_df.loc[i, 'Reference Answer']}")
    print("-"*50)
    for model_name in models_to_evaluate.keys():
        print(f"🤖 {model_name.upper()}:\n{qualitative_df.loc[i, f'{model_name} Prediction']}")
        print("-"*20)

# Save qualitative comparison file
qualitative_csv_path = "../reports/qualitative_comparison_examples.csv"
qualitative_df.to_csv(qualitative_csv_path, index=False, encoding='utf-8-sig')
print(f"\nSaved comparison examples to: {qualitative_csv_path}")